In [0]:
# --- カタログ・スキーマ・ボリューム ---
CATALOG = "your_catalog"        # 任意のカタログ名　ご自身の環境に合わせて変更してください
SCHEMA = "ai_functions_demo"    # スキーマ（なければ新規で作られる）
VOLUME = "files"

# --- モデル（Serving 画面にあるエンドポイント名を指定）---
LLM_ENDPOINT = "databricks-claude-sonnet-4-6"
# LLM_ENDPOINT = "databricks-gpt-oss-120b"
EMBEDDING_ENDPOINT = "databricks-qwen3-embedding-0-6b"   # 多言語対応（日本語可）
# EMBEDDING_ENDPOINT = "databricks-gte-large-en"         # 英語のみ

# --- AI Search（41_vector_search / 42_ai_search で使う）---
# エンドポイントがなければ 41_vector_search が新しく作る。⚠️ エンドポイントはインデックスを作った時点から、検索していなくても料金がかかる（最後のインデックスを消して24時間で止まる。2026-11-01時点）
# 使い終わったら、2つ目のセル「作成したリソースの削除」で削除する
VS_ENDPOINT = "konomi_demo_vs_endpoint"
VS_INDEX = f"{CATALOG}.{SCHEMA}.faq_articles_index"

# --- SQL ウェアハウス（02_Python_SQL / 03_Python の 53_ai_forecast・54_ai_detect_anomalies だけで使う）---
# ai_forecast と ai_detect_anomalies は SQL ウェアハウスでしか動かないため、Python の版はここに SQL を送って実行する
# 空のままなら、使える Serverless / Pro の SQL ウェアハウスを自動で選ぶ。決まったものを使うときだけ ID を書く
WAREHOUSE_ID = ""

# --- 環境バージョンの確認（AI 関数の結果の VARIANT 型を受け取るには、サーバレスの環境バージョン 3 以上が必要）---
try:
    spark.sql("SELECT parse_json('{}') AS v").collect()
except Exception:
    raise RuntimeError("サーバレスの環境バージョンが古いため、AI 関数の結果（VARIANT 型）を受け取れません。"
                       "右側の「環境」パネルで環境バージョンを 3 以上（推奨: 最新）にして、もう一度実行してください。")

# --- カタログ・スキーマ・ボリュームの初期化 ---
# spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
VOL_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# --- SQL セルから使う変数（%sql の中で llm_endpoint などと書ける）---
spark.sql(f"DECLARE OR REPLACE VARIABLE vol_root STRING DEFAULT '{VOL_ROOT}'")
spark.sql(f"DECLARE OR REPLACE VARIABLE llm_endpoint STRING DEFAULT '{LLM_ENDPOINT}'")
spark.sql(f"DECLARE OR REPLACE VARIABLE embedding_endpoint STRING DEFAULT '{EMBEDDING_ENDPOINT}'")
spark.sql(f"DECLARE OR REPLACE VARIABLE vs_index STRING DEFAULT '{VS_INDEX}'")

# --- SQL ウェアハウスの自動選択（WAREHOUSE_ID が空のとき）---
if not WAREHOUSE_ID:
    from databricks.sdk import WorkspaceClient
    _whs = [w for w in WorkspaceClient().warehouses.list()
            if w.enable_serverless_compute or str(w.warehouse_type).endswith("PRO")]
    _whs.sort(key=lambda w: (not w.enable_serverless_compute, str(w.state) != "State.RUNNING"))
    WAREHOUSE_ID = _whs[0].id if _whs else "（Serverless / Pro の SQL ウェアハウスが見つかりません）"

# --- サンプルファイル（assets フォルダの PDF・画像・音声）をボリュームにコピー ---
import os, shutil
_here = "/Workspace" + os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
ASSETS_DIR = next(d for d in (f"{_here}/assets", f"{os.path.dirname(_here)}/assets") if os.path.isdir(d))
for _sub in sorted(os.listdir(ASSETS_DIR)):
    os.makedirs(f"{VOL_ROOT}/{_sub}", exist_ok=True)
    for _f in os.listdir(f"{ASSETS_DIR}/{_sub}"):
        if not os.path.exists(f"{VOL_ROOT}/{_sub}/{_f}"):
            shutil.copy(f"{ASSETS_DIR}/{_sub}/{_f}", f"{VOL_ROOT}/{_sub}/{_f}")

print(f"✅ Catalog:    {CATALOG}")
print(f"✅ Schema:     {SCHEMA}")
print(f"✅ Volume:     {VOL_ROOT}")
print(f"✅ LLM:        {LLM_ENDPOINT}")
print(f"✅ Embedding:  {EMBEDDING_ENDPOINT}")
print(f"✅ VS:         {VS_ENDPOINT}")
print(f"✅ VS index:   {VS_INDEX}")
print(f"✅ Warehouse:  {WAREHOUSE_ID}")

In [0]:
# ⚠️⚠️⚠️ 注意: このセルは、このノートブック集で作ったものを「すべて削除」します ⚠️⚠️⚠️
#
# 使うのは、試し終わって片付けたいときだけです。ふだんはコメントアウトのままにしておいてください。
# 実行するときは、下の「削除するもの」で消したい行のコメント（行頭の #）を外し、このセルだけを実行します。
# 削除すると元に戻せません。
#
# このノートブック集を動かすと作られるもの（料金がかかるものに 💰）:
#   - スキーマ {CATALOG}.{SCHEMA} の中のテーブル（サンプルデータと結果）とボリューム {VOLUME}（PDF・画像・音声のコピー）
#   - 💰 AI Search のインデックス {VS_INDEX}（41_vector_search で作成）
#   - 💰 AI Search のエンドポイント {VS_ENDPOINT}（なかった場合だけ 41_vector_search が作成。インデックスを作った時点から、検索していなくても料金がかかる。最後のインデックスを消して24時間で止まる）
# 作られないもの: SQL ウェアハウス、モデルのエンドポイント（既存のものを使うだけ。AI 関数の料金は呼んだ分だけかかる）
#
# 安全のため、00_config を直接開いて実行したときだけ動きます（ほかのノートブックの %run ../00_config からは動きません）。

# _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# if _nb.endswith("/00_config"):
#     from databricks.sdk import WorkspaceClient
#     _w = WorkspaceClient()
#
#     # --- 削除するもの（消したい行だけコメントを外す）---
#
#     # 1. AI Search のインデックス
#     # _w.vector_search_indexes.delete_index(VS_INDEX)
#
#     # 2. AI Search のエンドポイント（⚠️ 41_vector_search が新しく作った場合だけ。ほかの用途でも使っているエンドポイントは消さない）
#     # _w.vector_search_endpoints.delete_endpoint(VS_ENDPOINT)
#
#     # 3. スキーマごと削除（⚠️ 中のテーブル・ボリューム・ファイルがすべて消える。このノートブック集のために新しく作ったスキーマのときだけ）
#     # spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")
#
#     print("🧹 削除しました")
# else:
#     print("このセルは 00_config を直接開いて実行したときだけ動きます")